In [ ]:
import pandas as pd
from pathlib import Path
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *
import scipy.io
from config import dir_config, main_config, ephys_config
from src.utils import ephys_utils
import pickle
from scipy.stats import ttest_rel


In [ ]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

session_metadata = pd.read_csv(processed_dir / "sessions_metadata.csv")

anticipatory_criteria = main_config.rejection_criteria.anticipatory_saccade
multi_saccade_criteria = main_config.rejection_criteria.multi_saccade

## Help functions

In [ ]:
def downsample_and_guassian_smooth(signal, original_rate=30000, target_rate=1000, sigma_ms=5):
    factor = original_rate // target_rate
    n_samples = signal.shape[0] // factor
    # vectorized mean pooling via reshape
    downsampled = np.nanmean(signal[:n_samples * factor].reshape(n_samples, factor), axis=1)
    sigma_samples = int(sigma_ms * target_rate / 1000)
    return scipy.ndimage.gaussian_filter1d(downsampled, sigma=sigma_samples)

def get_eye_data_for_session(session_id, timestamps, onset_event, pre_onset=0, post_onset=0):
    eye_parquet = compiled_dir / session_id / f"{session_id}_eye_tracking_processed.parquet"

    eye_signal = pd.read_parquet(eye_parquet, engine="fastparquet")

    ts = eye_signal.timestamp.values  # assumed sorted and regularly spaced
    ex = eye_signal.eye_x.values
    ey = eye_signal.eye_y.values
    del eye_signal

    # drop invalid trials
    timestamps = timestamps.dropna(subset=['response_onset'])

    if onset_event == "saccade":
        onsets = timestamps.response_onset.values
    elif onset_event == "go":
        onsets = timestamps.go_onset.values
    else:
        raise ValueError(f"Invalid onset_event: {onset_event}")

    window_length = (pre_onset + post_onset) * 30 + 1  # samples at 30kHz
    downsampled_length = window_length // 30
    n_trials = len(onsets)

    starts = np.searchsorted(ts, onsets - (pre_onset * 30))  # shape (n_trials,)

    # process one trial at a time -- avoids allocating (n_trials x window_length) arrays
    win_idx = np.arange(window_length)
    eye_data_processed = np.full((n_trials, downsampled_length), np.nan)
    for i in range(n_trials):
        idx = starts[i] + win_idx
        valid = idx < len(ts)
        idx_c = np.where(valid, idx, 0)
        wx = ex[idx_c]
        wy = ey[idx_c]
        wx[~valid] = np.nan
        wy[~valid] = np.nan
        dist = np.sqrt(wx**2 + wy**2)
        dist[~valid] = np.nan
        eye_data_processed[i, :] = downsample_and_guassian_smooth(dist, sigma_ms=5)

    return eye_data_processed.T

### filter out multi-saccade trials (i.e. fail to reach target within 75ms after saccade onset)

In [ ]:
percentage_threshold = multi_saccade_criteria.percentage_threshold
fixation_window = multi_saccade_criteria.fixation_window_radius
post_saccade = multi_saccade_criteria.post_saccade_window
onset_event = multi_saccade_criteria.onset_event
pre_onset_duration = multi_saccade_criteria.pre_onset_duration
post_onset_duration = multi_saccade_criteria.post_onset_duration
post_saccade_onset_duration = multi_saccade_criteria.post_saccade_onset_duration


trials_to_exclude = {}

for session_id in session_metadata.session_id.unique():
    print(f"Session {session_id}:")
    RF_coor = session_metadata.loc[session_metadata.session_id == session_id, "RF_coordinate"].values[0]
    # convert RF_coor string to tuple of floats
    RF_coor = tuple(map(float, RF_coor.strip("()").split(",")))
    RF_mag = np.sqrt(RF_coor[0]**2 + RF_coor[1]**2)

    timestamp_filename = compiled_dir / session_id / f"{session_id}_timestamps.csv"
    timestamps = pd.read_csv(timestamp_filename)

    eye_data_processed = get_eye_data_for_session(session_id, timestamps, onset_event, pre_onset=pre_onset_duration, post_onset=post_onset_duration)
    
    threshold = percentage_threshold * (RF_mag-fixation_window)  # Ensure a minimum threshold
    exclude = np.nanmax(eye_data_processed, axis=0) < threshold
    print(f"{exclude.sum()} / {len(exclude)} ({exclude.sum()/len(exclude):.2%}) trials excluded (max dist < threshold {threshold:.2f})")
    trials_to_exclude[session_id] = exclude
    del eye_data_processed
    

# # save trials_to_exclude dict
# with open(processed_dir / "trials_to_exclude.pkl", "wb") as f:
#     pickle.dump(trials_to_exclude, f)

In [ ]:
with open(processed_dir / "trials_to_exclude.pkl", "rb") as f:
    trials_to_exclude = pickle.load(f)

In [ ]:
for session_id in trials_to_exclude:
    print(f"Session {session_id}: {len(trials_to_exclude[session_id])} trials, {trials_to_exclude[session_id].sum()} to exclude ({trials_to_exclude[session_id].sum()/len(trials_to_exclude[session_id]):.2%})")

In [ ]:
# trials_to_exclude[session_id] is from non-nan reaction time trials, but i want to get trial number of these trials in the original trial_info dataframe, so i need to get the indices of non-nan reaction time trials and then apply the exclude mask to those indices to get the final trial numbers to exclude
# temp_sessions_ids = ['210312_GP_JP', "210602_GP_JP", "210603_GP_JP", "240805_GP_TZ", "240814_GP_TZ", "240828_GP_TZ", "240903_GP_TZ", "241002_GP_TZ"]
# for session_id in temp_sessions_ids:
for session_id in session_metadata.session_id:
    # load trial_info and timestamps for the session
    trial_info = pd.read_csv(compiled_dir / session_id / f"{session_id}_trial.csv")
    timestamp_filename = compiled_dir / session_id / f"{session_id}_timestamps.csv"
    timestamps = pd.read_csv(timestamp_filename)

    valid_trial_indices = np.where(~np.isnan(trial_info.reaction_time))[0]
    final_exclude_indices = valid_trial_indices[trials_to_exclude[session_id]]
    
    # set those trial reaction time to nan in the trial_info dataframe
    trial_info.loc[final_exclude_indices, "reaction_time"] = np.nan
    # set response_onset in timestamps to nan for those trials as well
    
    timestamps.loc[final_exclude_indices, "response_onset"] = np.nan

    # I want to plot the eye_data after excluding those trials
    # eye_data_processed = get_eye_data_for_session(session_id, timestamps,post_saccade=200)

    eye_data_processed = get_eye_data_for_session(session_id, timestamps, onset_event="saccade", pre_onset=100, post_onset=200)
    
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(session_id, fontsize=14)

    axs[0].plot(eye_data_processed, color="blue", alpha=0.2)
    axs[0].axvline(101, color="k", linestyle="--", label="saccade onset")
    axs[0].set_ylabel("Position")

    axs[1].plot(1000 * np.diff(eye_data_processed, axis=0), color="red", alpha=0.2)
    axs[1].axvline(100, color="k", linestyle="--", label="saccade onset")
    axs[1].set_ylabel("Velocity")

    axs[2].plot(1e6 * np.diff(np.diff(eye_data_processed, axis=0), axis=0), color="green", alpha=0.2)
    axs[2].axvline(99, color="k", linestyle="--", label="saccade onset")
    axs[2].set_ylabel("Acceleration")

    plt.tight_layout()
    plt.show()

    del eye_data_processed
    
    

### filter out anticipatory saccade trials

In [ ]:
# trials_to_exclude[session_id] is from non-nan reaction time trials, but i want to get trial number of these trials in the original trial_info dataframe, so i need to get the indices of non-nan reaction time trials and then apply the exclude mask to those indices to get the final trial numbers to exclude
# temp_sessions_ids = ['210312_GP_JP', "210602_GP_JP", "210603_GP_JP", "240805_GP_TZ", "240814_GP_TZ", "240828_GP_TZ", "240903_GP_TZ", "241002_GP_TZ"]
# temp_sessions_ids = ['210528_GP_JP']

# anticipatory_time_window = 90 # upto 90ms after go cue
# anticipatory_amp_threshold = 3 # 3 degree visual angle
# anticipatory_trials = {}
# pre_onset = 200

anticipatory_time_window = anticipatory_criteria.post_onset_duration
anticipatory_amp_threshold = anticipatory_criteria.amplitude_threshold
pre_onset = anticipatory_criteria.pre_onset_duration


# for session_id in temp_sessions_ids:
for session_id in session_metadata.session_id:
    # load trial_info and timestamps for the session
    trial_info = pd.read_csv(compiled_dir / session_id / f"{session_id}_trial.csv")
    timestamp_filename = compiled_dir / session_id / f"{session_id}_timestamps.csv"
    timestamps = pd.read_csv(timestamp_filename)

    valid_trial_indices = np.where(~np.isnan(trial_info.reaction_time))[0]
    final_exclude_indices = valid_trial_indices[trials_to_exclude[session_id]]
    
    # set those trial reaction time to nan in the trial_info dataframe
    trial_info.loc[final_exclude_indices, "reaction_time"] = np.nan
    # set response_onset in timestamps to nan for those trials as well
    
    timestamps.loc[final_exclude_indices, "response_onset"] = np.nan

    # I want to plot the eye_data after excluding those trials
    # eye_data_processed = get_eye_data_for_session(session_id, timestamps,post_saccade=200)
    eye_data_processed = get_eye_data_for_session(session_id, timestamps, onset_event="go", pre_onset=pre_onset, post_onset=500)

    
    # rejection 
    anticipatory_mask = np.nanmax(eye_data_processed[:pre_onset+anticipatory_time_window,:], axis=0) > anticipatory_amp_threshold
    anticipatory_trials[session_id] = anticipatory_mask


    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(session_id, fontsize=14)

    axs[0].plot(eye_data_processed, color="blue", alpha=0.2)
    axs[0].plot(eye_data_processed[:, anticipatory_mask], color="orange", alpha=0.5, label="anticipatory trials")
    axs[0].axvline(pre_onset+1, color="k", linestyle="--", label="saccade onset")
    axs[0].set_ylabel("Position")

    axs[1].plot(1000 * np.diff(eye_data_processed, axis=0), color="red", alpha=0.2)
    axs[1].plot(1000 * np.diff(eye_data_processed[:, anticipatory_mask], axis=0), color="orange", alpha=0.5)
    axs[1].axvline(pre_onset, color="k", linestyle="--", label="saccade onset")
    axs[1].set_ylabel("Velocity")

    axs[2].plot(1e6 * np.diff(np.diff(eye_data_processed, axis=0), axis=0), color="green", alpha=0.2)
    axs[2].plot(1e6 * np.diff(np.diff(eye_data_processed[:, anticipatory_mask], axis=0), axis=0), color="orange", alpha=0.5)
    axs[2].axvline(pre_onset-1, color="k", linestyle="--", label="saccade onset")
    axs[2].set_ylabel("Acceleration")

    plt.tight_layout()
    plt.show()

    del eye_data_processed
    

In [ ]:
percentage_threshold = multi_saccade_criteria.percent_target_threshold
fixation_window = multi_saccade_criteria.fixation_window_radius
anticipatory_amp_threshold = anticipatory_criteria.amplitude_threshold

trials_to_exclude = {}

for session_id in session_metadata.session_id.unique():
    print(f"Session {session_id}:")
    RF_coor = session_metadata.loc[session_metadata.session_id == session_id, "RF_coordinate"].values[0]
    RF_coor = tuple(map(float, RF_coor.strip("()").split(",")))
    RF_mag = np.sqrt(RF_coor[0]**2 + RF_coor[1]**2)

    timestamps = pd.read_csv(compiled_dir / session_id / f"{session_id}_timestamps.csv")

    # Multi-saccade filtering
    saccade_eye_data_processed = get_eye_data_for_session(session_id, timestamps, onset_event=multi_saccade_criteria.onset_event, 
                                                  pre_onset=multi_saccade_criteria.pre_onset_duration, post_onset=multi_saccade_criteria.post_onset_duration)
    threshold = percentage_threshold * (RF_mag - fixation_window)
    multi_saccade_mask = np.nanmax(saccade_eye_data_processed, axis=0) < threshold
    print(f"Multi-saccade: {multi_saccade_mask.sum()} / {len(multi_saccade_mask)} ({multi_saccade_mask.sum()/len(multi_saccade_mask):.2%}) trials excluded (max dist < threshold {threshold:.2f})")


    # Anticipatory saccade filtering
    go_eye_data_processed = get_eye_data_for_session(session_id, timestamps, onset_event=anticipatory_criteria.onset_event, 
                                                  pre_onset=anticipatory_criteria.pre_onset_duration, post_onset=anticipatory_criteria.post_onset_duration)
    anticipatory_mask = np.nanmax(go_eye_data_processed, axis=0) > anticipatory_amp_threshold
    print(f"Anticipatory: {anticipatory_mask.sum()} / {len(anticipatory_mask)} ({anticipatory_mask.sum()/len(anticipatory_mask):.2%}) trials excluded ")


    trials_to_exclude[session_id] = multi_saccade_mask | anticipatory_mask
    print(f"Combined: {trials_to_exclude[session_id].sum()} / {len(trials_to_exclude[session_id])} ({trials_to_exclude[session_id].sum()/len(trials_to_exclude[session_id]):.2%}) trials excluded")
    
    
    


In [ ]:
for session_id in session_metadata.session_id.unique():
    print(f"Session {session_id}:")
    timestamps = pd.read_csv(compiled_dir / session_id / f"{session_id}_timestamps.csv")

    saccade_eye_data_processed = get_eye_data_for_session(session_id, timestamps, onset_event=multi_saccade_criteria.onset_event, 
                                                  pre_onset=multi_saccade_criteria.pre_onset_duration, post_onset=200)
    
    go_eye_data_processed = get_eye_data_for_session(session_id, timestamps, onset_event=anticipatory_criteria.onset_event, 
                                                  pre_onset=anticipatory_criteria.pre_onset_duration, post_onset=400)
    
# plot saccade_eye_data_processed excluding trials marked for exclusion
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(session_id, fontsize=14)
    axs[0].plot(saccade_eye_data_processed[:, ~trials_to_exclude[session_id]], color="blue", alpha=0.2)
    axs[0].plot(saccade_eye_data_processed[:, trials_to_exclude[session_id]], color="orange", alpha=0.5, label="excluded trials")
    axs[0].axvline(multi_saccade_criteria.pre_onset_duration+1, color="k", linestyle="--", label="saccade onset")
    axs[1].plot(1000 * np.diff(saccade_eye_data_processed[:, ~trials_to_exclude[session_id]], axis=0), color="red", alpha=0.2)
    axs[1].plot(1000 * np.diff(saccade_eye_data_processed[:, trials_to_exclude[session_id]], axis=0), color="orange", alpha=0.5)
    axs[1].axvline(multi_saccade_criteria.pre_onset_duration, color="k", linestyle="--", label="saccade onset")
    axs[2].plot(1e6 * np.diff(np.diff(saccade_eye_data_processed[:, ~trials_to_exclude[session_id]], axis=0), axis=0), color="green", alpha=0.2)
    axs[2].plot(1e6 * np.diff(np.diff(saccade_eye_data_processed[:, trials_to_exclude[session_id]], axis=0), axis=0), color="orange", alpha=0.5)
    axs[2].axvline(multi_saccade_criteria.pre_onset_duration-1, color="k", linestyle="--", label="saccade onset")
    fig.suptitle('Aligned to saccade onset')
    plt.tight_layout()
    plt.show()

    # plot go_eye_data_processed excluding trials marked for exclusion
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    axs[0].plot(go_eye_data_processed[:, ~trials_to_exclude[session_id]], color="blue", alpha=0.2)
    axs[0].plot(go_eye_data_processed[:, trials_to_exclude[session_id]], color="orange", alpha=0.5, label="excluded trials")
    axs[0].axvline(anticipatory_criteria.pre_onset_duration+1, color="k", linestyle="--", label="go onset")
    axs[1].plot(1000 * np.diff(go_eye_data_processed[:, ~trials_to_exclude[session_id]], axis=0), color="red", alpha=0.2)
    axs[1].plot(1000 * np.diff(go_eye_data_processed[:, trials_to_exclude[session_id]], axis=0), color="orange", alpha=0.5)
    axs[1].axvline(anticipatory_criteria.pre_onset_duration, color="k", linestyle="--", label="go onset")
    axs[2].plot(1e6 * np.diff(np.diff(go_eye_data_processed[:, ~trials_to_exclude[session_id]], axis=0), axis=0), color="green", alpha=0.2)
    axs[2].plot(1e6 * np.diff(np.diff(go_eye_data_processed[:, trials_to_exclude[session_id]], axis=0), axis=0), color="orange", alpha=0.5)
    axs[2].axvline(anticipatory_criteria.pre_onset_duration-1, color="k", linestyle="--", label="go onset") 
    fig.suptitle('Aligned to go onset')
    plt.tight_layout()
    plt.show()

### Save new timestamps and trial_info

In [ ]:
for session_id in session_metadata.session_id:
    # convert trials_exclude to actual idx in timestamps
    timestamps = pd.read_csv(compiled_dir / session_id / f"{session_id}_timestamps.csv")
    trial_info = pd.read_csv(compiled_dir / session_id / f"{session_id}_trial.csv")

    # non-nan reaction time trial indices
    valid_trial_indices = np.where(~np.isnan(trial_info.reaction_time))[0]
    # actual trial indices to exclude in trial_info and timestamp
    final_exclude_indices = valid_trial_indices[trials_to_exclude[session_id]]
    
    # set those trial reaction time to nan in the trial_info dataframe
    trial_info.loc[final_exclude_indices, "reaction_time"] = np.nan
    # set response_onset in timestamps to nan for those trials as well
    timestamps.loc[final_exclude_indices, "response_onset"] = np.nan

    # save new timestamp and trial_info
    timestamps.to_csv(compiled_dir / session_id / f"{session_id}_timestamps_cleaned.csv")
    trial_info.to_csv((compiled_dir / session_id / f"{session_id}_trial_cleaned.csv"))

In [ ]:
for session_id in session_metadata.session_id.unique():

    timestamps_orig = pd.read_csv(compiled_dir / session_id / f"{session_id}_timestamps.csv")
    print(f"Session {session_id}:")
    print(f"Before cleaning: {timestamps_orig.shape[0]} trials with {timestamps_orig['response_onset'].isna().sum()} NaN RT")

    # Step 1: non-NaN RT in original → 1247 indices
    has_rt_idx = np.where(~timestamps_orig['response_onset'].isna().values)[0]  # (1247,)

    # Step 2: apply trials_to_exclude → 837 indices into original timestamps
    passed_exclude_idx = has_rt_idx[~trials_to_exclude[session_id]]              # (837,)

    # Step 3: apply anticipatory_trials → final good indices
    good_idx = passed_exclude_idx[~anticipatory_trials[session_id]]

    # Boolean mask aligned to original timestamps (1443,)
    good_trials_mask = np.zeros(len(timestamps_orig), dtype=bool)
    good_trials_mask[good_idx] = True
    # made response_onset NaN for bad trials in original timestamps
    timestamps_orig.loc[~good_trials_mask, 'response_onset'] = np.nan

    print(f"After cleaning: {timestamps_orig.shape[0]} trials with {timestamps_orig['response_onset'].isna().sum()} NaN RT")

    # Optional: save the updated timestamps as session_id + "_timestamps_cleaned.csv"
    timestamps_orig.to_csv(compiled_dir / session_id / f"{session_id}_timestamps_cleaned.csv", index=False)

===========================